# Visualise Plots

In [7]:
import pdal
import pandas as pd
import numpy as np
import geopandas as gpd

import pyvista as pv
pv.set_jupyter_backend('client')

from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

In [8]:
plot_lidar_dir = Path("../data/outputs/plots/lidar")
plots_gdf = gpd.read_file("../data/outputs/plots/plots.geojson")
# plots_gdf
plots = plots_gdf["id"].to_list()

In [9]:
def read_plot(plot_id: str):
    plot_path = plot_lidar_dir / f"{plot_id}.copc.laz"
    pl = pdal.Reader(str(plot_path)).pipeline()
    pl.execute()
    return pl.arrays[0]

def render_point_cloud(points, color='height', width=1200, height=900, font_size=18, point_size=3):
    X = points["X"] - points["X"].min()
    Y = points["Y"] - points["Y"].min()
    Z = points["Z"]
    positions = np.column_stack((X, Y, Z)).astype("float32")

    R = (points["Red"] / 65535 * 255).astype("uint8")
    G = (points["Green"] / 65535 * 255).astype("uint8")
    B = (points["Blue"] / 65535 * 255).astype("uint8")
    colors = np.column_stack((R, G, B))

    mesh = pv.PolyData(positions)
    mesh['height'] = Z
    mesh['rgb'] = colors

    # Create a Plotter with a larger window size for taller visualization
    plotter = pv.Plotter(notebook=True, window_size=(width, height))

    if color == 'rgb':
        # add as RGB colors
        plotter.add_mesh(mesh, scalars='rgb', rgb=True, point_size=point_size, render_points_as_spheres=True)
    else:
        # color by height with a scalar bar whose text is larger
        plotter.add_mesh(mesh, scalars='height', cmap='viridis', point_size=point_size, render_points_as_spheres=True)
        plotter.add_scalar_bar(title='height', title_font_size=font_size, label_font_size=font_size)

    # Show a grid by default
    # plotter.show_bounds()

    # Optional label in corner with larger text
    # plotter.add_text("Plot view", position='upper_left', font_size=font_size, color='white')

    # Show using the jupyter 'client' backend you set earlier
    plotter.show(jupyter_backend='client')


def read_and_render_plot(plot_id: str, color='height', width=1000, height=900, font_size=18, point_size=3):
    points = read_plot(plot_id)
    render_point_cloud(points, color=color, width=width, height=height, font_size=font_size, point_size=point_size)
# ...existing code...


In [10]:
# Create interactive widgets for site selection
plot_dropdown = widgets.Dropdown(
    options=plots,
    value=plots[0] if plots else None,
    description='plot:',
    disabled=False,
)

color_dropdown = widgets.Dropdown(
    options=['rgb', 'height'],
    value='height',
    description='Color by:',
    disabled=False,
)

# Function to handle the plot update
def update_plot(plot, color):
    read_and_render_plot(plot, color=color, point_size=5)

# Create interactive plot
interactive_plot = widgets.interactive(update_plot, plot=plot_dropdown, color=color_dropdown)
display(interactive_plot)

interactive(children=(Dropdown(description='plot:', options=('AGG_O_01_P1', 'AGG_O_01_P2', 'AGG_O_01_P3', 'AGG…